In [ ]:
import random
import numpy as np
import pandas as pd
import networkx as nx
from cellmaps_ppi_embedding.runner import Node2VecEmbeddingGenerator

import sys
sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import concat_emb

### run n2v to get input embedding

In [ ]:
def run_n2v(infile, outfile, seed=1):
    random.seed(seed)
    np.random.seed(seed)

    df = pd.read_csv(infile, sep="\t")
    g = nx.Graph()
    g.add_edges_from(df[['GeneA', 'GeneB']].values.tolist())
    
    # parameters choosen from 1.3_n2v_grid/n2v_gridsearch_summary
    gen = Node2VecEmbeddingGenerator(
        nx_network=g,
        dimensions=128,
        p=0.25,
        q=0.25,
        walk_length=320,
        num_walks=40,
        workers=1)
    
    result = []
    for row in gen.get_next_embedding():
        result.append(row)
    pd.DataFrame(result).to_csv(outfile, sep="\t", index=False, header=True)
    return

In [ ]:
run_n2v(
    "../../result/HEK293T_HCT116/input/BioPlex-HEK293T.ppi",
    "../../result/HEK293T_HCT116/input/BioPlex-HEK293T.tsv"
)

run_n2v(
    "../../result/HEK293T_HCT116/input/BioPlex-HCT116.ppi",
    "../../result/HEK293T_HCT116/input/BioPlex-HCT116.tsv"
)

run_n2v(
    "../../result/HEK293T_HCT116/input/SECMX-HEK293T.ppi",
    "../../result/HEK293T_HCT116/input/SECMX-HEK293T.tsv"
)

run_n2v(
    "../../result/HEK293T_HCT116/input/SECMX-HCT116.ppi",
    "../../result/HEK293T_HCT116/input/SECMX-HCT116.tsv"
)

### Stats for future filtering

In [ ]:
# load genes in each experiment
ap_hct = set(pd.read_csv("../../result/HEK293T_HCT116/input/BioPlex-HCT116.tsv", sep="\t", index_col=0).index)
ap_hek = set(pd.read_csv("../../result/HEK293T_HCT116/input/BioPlex-HEK293T.tsv", sep="\t", index_col=0).index)
sec_hct = set(pd.read_csv("../../result/HEK293T_HCT116/input/SECMX-HCT116.tsv", sep="\t", index_col=0).index)
sec_hek = set(pd.read_csv("../../result/HEK293T_HCT116/input/SECMX-HEK293T.tsv", sep="\t", index_col=0).index)

In [ ]:
# total possible pairs in each cell line
hct_genes = ap_hct | sec_hct
hct_pairs = len(hct_genes) * (len(hct_genes)-1) // 2

hek_genes = ap_hek | sec_hek
hek_pairs = len(hek_genes) * (len(hek_genes)-1) // 2

In [ ]:
# top 1%, add other when needed
stat = pd.DataFrame(
    [
        [int(hct_pairs*0.01), int(hek_pairs*0.01)]
    ],
    index=["Percent1"],
    columns=["HCT116", "HEK293T"]
)
stat.to_csv("../../result/HEK293T_HCT116/input/pair_stat.tsv", sep="\t")

### concat the n2v embedding from modalities

In [ ]:
df1 = pd.read_csv("../../result/HEK293T_HCT116/input/BioPlex-HCT116.tsv", sep="\t", index_col=0)
df2 = pd.read_csv("../../result/HEK293T_HCT116/input/SECMX-HCT116.tsv", sep="\t", index_col=0)
df_concat = concat_emb([df1, df2])
df_concat.to_csv("../../result/HEK293T_HCT116/input/Concat-HCT116.tsv", sep="\t", header=True, index=True)

In [ ]:
df1 = pd.read_csv("../../result/HEK293T_HCT116/input/BioPlex-HEK293T.tsv", sep="\t", index_col=0)
df2 = pd.read_csv("../../result/HEK293T_HCT116/input/SECMX-HEK293T.tsv", sep="\t", index_col=0)
df_concat = concat_emb([df1, df2])
df_concat.to_csv("../../result/HEK293T_HCT116/input/Concat-HEK293T.tsv", sep="\t", header=True, index=True)